# Stable Diffusion with Hugging Face
This notebook provides an overview of Stable Diffusion models, their architecture, and use cases such as Text-to-Image, Inpainting, and ControlNets.

## Introduction
Stable Diffusion is a state-of-the-art generative model for creating high-quality images from text prompts. It leverages diffusion processes to iteratively refine images, producing realistic and detailed outputs.

## Architecture Overview
Stable Diffusion models are based on a diffusion process that gradually transforms a simple noise distribution into a complex data distribution, such as images. The model consists of a series of denoising autoencoders that iteratively refine the image.

## Use Cases
- **Text-to-Image**: Generate images from textual descriptions.
- **Inpainting**: Fill in missing parts of an image.
- **ControlNets**: Use additional control signals to guide the image generation process.

## Setup
Install the necessary libraries and set up the environment.

In [ ]:
!pip install diffusers transformers torch

## Importing the Model
Use the Hugging Face `diffusers` library to import a Stable Diffusion model.

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

# Load the model
model_id = 'CompVis/stable-diffusion-v1-4'
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to('cuda')  # Use GPU for faster inference

## Text-to-Image Example
Generate an image from a text prompt.

In [ ]:
prompt = 'A fantasy landscape with mountains and a river'
image = pipe(prompt).images[0]
image.show()

## Inpainting Example
Fill in missing parts of an image using a mask.

In [ ]:
from diffusers import StableDiffusionInpaintPipeline
from PIL import Image
import requests
from io import BytesIO

# Load the inpainting model
inpaint_model_id = "runwayml/stable-diffusion-inpainting"
inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
    inpaint_model_id,
    torch_dtype=torch.float16
)
inpaint_pipe = inpaint_pipe.to("cuda")

# Load an example image and create a simple mask
url = "https://raw.githubusercontent.com/CompVis/latent-diffusion/main/data/inpainting_examples/overture-creations-5sI6fQgYIuo.png"
response = requests.get(url)
init_image = Image.open(BytesIO(response.content)).convert("RGB")
init_image = init_image.resize((512, 512))

# Create a white mask (to be inpainted) in the center
mask_image = Image.new("RGB", (512, 512), "black")
mask_width = 128
mask_height = 128
x = (512 - mask_width) // 2
y = (512 - mask_height) // 2
mask_image.paste("white", (x, y, x + mask_width, y + mask_height))
mask_image = mask_image.convert("RGB")

# Generate the inpainted image
prompt = "a cat sitting in the middle"
image = inpaint_pipe(
    prompt=prompt,
    image=init_image,
    mask_image=mask_image,
).images[0]

# Display the original, mask, and result
init_image.show()
mask_image.show()
image.show()

## ControlNets Example
Guide the image generation process using control signals.

In [ ]:
# Install required packages
# !pip install controlnet_aux diffusers==0.18.2

from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from diffusers.utils import load_image
import torch
import numpy as np
from PIL import Image

# Load the canny edge detection controlnet model
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny",
    torch_dtype=torch.float16
)

# Create pipeline with controlnet
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
)
pipe = pipe.to("cuda")

# Load and prepare image
init_image = load_image(
    "https://raw.githubusercontent.com/CompVis/latent-diffusion/main/data/inpainting_examples/overture-creations-5sI6fQgYIuo.png"
)
init_image = init_image.resize((512, 512))

# Get canny edge detection image
from controlnet_aux import CannyDetector
canny = CannyDetector()
control_image = canny(init_image, low_threshold=100, high_threshold=200)

# Generate image with controlnet guidance
prompt = "a cat sitting in the middle"
image = pipe(
    prompt,
    control_image,
    num_inference_steps=20
).images[0]

# Display results
init_image.show()
control_image.show() 
image.show()